In [ ]:
"""
Scalable session-level conversation experiments with LLM-as-a-Judge evaluation
"""

from langfuse import get_client, propagate_attributes
from langfuse.openai import OpenAI
from agents.chatbot import LlamaChatbot
from agents.simulator import SimulatedUser
from metrics.metrics2 import SessionEvaluator
from utils.utils2 import load_config, validate_langfuse_connection, SessionManager, get_logger
from typing import Dict, List
import sys
import time

# Initialize logger
logger = get_logger(__name__)

# Load configuration
try:
    config = load_config("config/config.json")
    logger.info("✓ Configuration loaded successfully")
except Exception as e:
    logger.error(f"Failed to load configuration: {e}")
    sys.exit(1)

# Initialize Langfuse client
langfuse = get_client()

# Verify connection
if not validate_langfuse_connection(langfuse):
    logger.error("❌ Langfuse authentication failed")
    sys.exit(1)
logger.info("✅ Langfuse client authenticated and ready")

# Initialize evaluation metrics
try:
    evaluator = SessionEvaluator(
        judge_model=config['models']['judge_model'],
        ollama_config=config['ollama'],
        evaluator_config=config['evaluators'],
        logger=logger
    )
    logger.info("✓ Session evaluator initialized")
except Exception as e:
    logger.error(f"Failed to initialize evaluator: {e}")
    sys.exit(1)

def simulate_single_turn(
    langfuse_client,
    chatbot: LlamaChatbot,
    simulated_user: SimulatedUser,
    turn_number: int,
    session_id: str,
    persona: str,
    scenario: str,
    is_first: bool = False
) -> Dict:
    """Simulate a single conversation turn as a separate trace"""
    try:
        trace_id = SessionManager.generate_trace_id(langfuse_client, session_id, turn_number)
        
        with langfuse_client.start_as_current_observation(
            as_type="span",
            name=f"Turn {turn_number}",
            trace_context={"trace_id": trace_id}
        ) as span:
            with propagate_attributes(session_id=session_id):
                user_message = simulated_user.generate_message(is_first_turn=is_first)
                assistant_response = chatbot.chat(user_message, turn_number)
                simulated_user.update_history(user_message, assistant_response)
                
                langfuse_client.update_current_trace(
                    name=f"Turn {turn_number}",
                    input={"user_message": user_message},
                    output={"assistant_response": assistant_response},
                    metadata={
                        "turn_number": turn_number,
                        "persona": persona,
                        "scenario": scenario
                    }
                )
                
                return {
                    "turn": turn_number,
                    "user": user_message,
                    "assistant": assistant_response,
                    "trace_id": trace_id
                }
    
    except Exception as e:
        logger.error(f"Error in turn {turn_number}: {e}", exc_info=True)
        return {
            "turn": turn_number,
            "user": "",
            "assistant": f"Error: {str(e)}",
            "trace_id": None
        }

def simulate_continuous_conversation(
    langfuse_client,
    persona: str,
    scenario: str,
    session_id: str,
    max_turns: int = 6
) -> Dict:
    """Simulate a continuous conversation with each turn as a separate trace"""
    conversation_log = []
    
    try:
        # Validate inputs
        if not persona or not scenario:
            logger.error(f"Invalid inputs - persona: {bool(persona)}, scenario: {bool(scenario)}")
            return {
                "conversation_log": [],
                "session_id": session_id,
                "num_turns": 0,
                "error": "Missing persona or scenario"
            }
        
        chatbot = LlamaChatbot(
            model=config['models']['answer_model'],
            ollama_config=config['ollama'],
            session_id=session_id
        )
        
        simulated_user = SimulatedUser(
            persona=persona,
            scenario=scenario,
            model=config['models']['answer_model_alt'],
            ollama_config=config['ollama']
        )
        
        SessionManager.log_session_start(session_id, persona, scenario)
        logger.info(f"🔄 Starting session: {session_id}")
        
        for turn in range(1, max_turns + 1):
            logger.info(f"--- Turn {turn}/{max_turns} ---")
            
            try:
                is_first = (turn == 1)
                turn_result = simulate_single_turn(
                    langfuse_client,
                    chatbot,
                    simulated_user,
                    turn,
                    session_id,
                    persona,
                    scenario,
                    is_first
                )
                
                # Only add turn if it has valid content
                if turn_result.get('user') and turn_result.get('assistant'):
                    conversation_log.append(turn_result)
                    logger.info(f"USER: {turn_result['user'][:100]}...")
                    logger.info(f"ASSISTANT: {turn_result['assistant'][:100]}...")
                else:
                    logger.warning(f"Turn {turn} produced empty content, skipping")
                    
            except Exception as turn_error:
                logger.error(f"Error in turn {turn}: {turn_error}", exc_info=True)
                # Continue to next turn instead of failing entire conversation
                continue
        
        logger.info(f"✅ Completed {len(conversation_log)} turns out of {max_turns} attempted")
        
        # Validate we have at least some conversation
        if not conversation_log:
            logger.error("No valid turns were generated in the conversation")
            return {
                "conversation_log": [],
                "session_id": session_id,
                "num_turns": 0,
                "error": "No valid conversation turns generated"
            }
        
        return {
            "conversation_log": conversation_log,
            "session_id": session_id,
            "num_turns": len(conversation_log),
            "persona": persona,
            "scenario": scenario
        }
    
    except Exception as e:
        logger.error(f"Critical error in conversation simulation: {e}", exc_info=True)
        return {
            "conversation_log": conversation_log,  # Return whatever we have
            "session_id": session_id,
            "num_turns": len(conversation_log),
            "error": str(e)
        }

def display_session_scores(score_results: List[Dict], weighted_average: float, session_id: str):
    """Display formatted session scores"""
    logger.info(f"\n{'='*80}")
    logger.info(f"📈 SESSION EVALUATION SCORES: {session_id}")
    logger.info(f"{'='*80}\n")
    
    logger.info("Individual Metric Scores:")
    logger.info("-" * 80)
    
    for idx, score_result in enumerate(score_results, 1):
        logger.info(f"\n[{idx}] {score_result['name']}")
        logger.info(f"    Score: {score_result['score']:.2f}/5.0")
        logger.info(f"    Reasoning: {score_result['reasoning'][:150]}...")
    
    logger.info(f"\n{'='*80}")
    logger.info(f"📊 OVERALL SESSION QUALITY: {weighted_average:.2f}/5.0")
    logger.info(f"{'='*80}\n")

def create_task_function(langfuse_client, evaluator_instance):
    """Factory function to create task with langfuse_client in closure"""
    def run_task(*, item, **kwargs) -> Dict:
        """Task function for Langfuse experiment with session-level evaluation"""
        try:
            persona = item.input.get("persona", "")
            scenario = item.input.get("scenario", "")
            
            # Validate inputs early
            if not persona or not scenario:
                logger.error(f"Dataset item {item.id} missing persona or scenario")
                return {
                    "session_id": "error",
                    "num_turns": 0,
                    "conversation_log": [],
                    "error": "Missing persona or scenario in dataset item"
                }
            
            session_id = SessionManager.generate_session_id(item.id, prefix="exp")
            
            logger.info(f"\n{'='*80}")
            logger.info(f"🔄 Processing dataset item: {item.id}")
            logger.info(f"📍 Session ID: {session_id}")
            logger.info(f"👤 Persona: {persona[:80]}...")
            logger.info(f"📝 Scenario: {scenario[:80]}...")
            logger.info(f"{'='*80}\n")
            
            # Simulate the full conversation
            result = simulate_continuous_conversation(
                langfuse_client,
                persona=persona,
                scenario=scenario,
                session_id=session_id,
                max_turns=config['dataset']['max_turns']
            )
            
            # Check for errors in simulation
            if result.get('error'):
                logger.error(f"Simulation error: {result['error']}")
                return {
                    "session_id": session_id,
                    "num_turns": result.get('num_turns', 0),
                    "conversation_log": result.get('conversation_log', []),
                    "error": result['error']
                }
            
            # Validate conversation log
            conversation_log = result.get('conversation_log', [])
            if not conversation_log:
                logger.error(f"No conversation log found for session {session_id}")
                logger.error(f"Result keys: {result.keys()}")
                logger.error(f"Result: {result}")
                return {
                    "session_id": session_id,
                    "num_turns": 0,
                    "conversation_log": [],
                    "error": "No conversation log generated"
                }
            
            logger.info(f"✓ Conversation log has {len(conversation_log)} turns")
            
            # STEP 1: Flush all traces first
            logger.info(f"⏳ Step 1: Flushing traces for session {session_id}")
            langfuse_client.flush()
            
            # STEP 2: Wait for backend processing
            logger.info(f"⏳ Step 2: Waiting for backend processing...")
            time.sleep(5)  # Increased wait time for backend
            
            # STEP 3: Evaluate and score the session
            session_scores = []
            logger.info(f"\n🔍 Step 3: Evaluating session {session_id}")
            
            try:
                # Evaluate the session
                session_eval = evaluator_instance.evaluate_session(
                    conversation_log=conversation_log,
                    persona=persona,
                    scenario=scenario
                )
                
                logger.info(f"📊 Step 4: Attaching scores to session {session_id}")
                
                # Attach individual metric scores using create_score with session_id
                score_count = 0
                for score_result in session_eval['individual_scores']:
                    try:
                        # Use create_score with session_id parameter
                        langfuse_client.create_score(
                            name=score_result['name'],
                            value=float(score_result['score']),
                            session_id=session_id,
                            data_type="NUMERIC",
                            comment=score_result['reasoning']
                        )
                        score_count += 1
                        session_scores.append({
                            "name": score_result['name'],
                            "value": float(score_result['score']),
                            "comment": score_result['reasoning']
                        })
                        logger.info(f"  ✓ [{score_count}] {score_result['name']}: {score_result['score']}/5")
                    except Exception as e:
                        logger.error(f"  ✗ Failed to score {score_result['name']}: {e}", exc_info=True)
                
                # Attach overall session quality score
                try:
                    overall_score = float(session_eval['weighted_average'])
                    langfuse_client.create_score(
                        name="overall_session_quality",
                        value=overall_score,
                        session_id=session_id,
                        data_type="NUMERIC",
                        comment=f"Weighted average: {session_eval['weighted_average']:.2f}/5"
                    )
                    score_count += 1
                    session_scores.append({
                        "name": "overall_session_quality",
                        "value": overall_score,
                        "comment": f"Weighted average: {session_eval['weighted_average']:.2f}/5"
                    })
                    logger.info(f"  ✓ [{score_count}] overall_session_quality: {session_eval['weighted_average']:.2f}/5")
                except Exception as e:
                    logger.error(f"  ✗ Failed to score overall_session_quality: {e}", exc_info=True)
                
                # STEP 5: Flush scores immediately
                logger.info(f"⏳ Step 5: Flushing {score_count} scores for session {session_id}")
                langfuse_client.flush()
                
                # STEP 6: Wait for scores to be processed
                logger.info(f"⏳ Step 6: Waiting for scores to be processed...")
                time.sleep(3)
                
                # STEP 7: Display all session scores
                logger.info(f"\n📊 Step 7: Displaying session evaluation results")
                display_session_scores(
                    session_eval['individual_scores'],
                    session_eval['weighted_average'],
                    session_id
                )
                
                logger.info(f"\n✅ Session {session_id} complete with {score_count} scores attached")
                logger.info(f"   View in Langfuse UI: Sessions → {session_id}")
                logger.info(f"   Or check Scores table filtered by session_id: {session_id}")
                
            except Exception as eval_error:
                logger.error(f"Error during evaluation: {eval_error}", exc_info=True)
                # Return partial results even if evaluation fails
            
            return {
                "session_id": session_id,
                "num_turns": result["num_turns"],
                "conversation_log": conversation_log,
                "persona": result.get("persona", persona),
                "scenario": result.get("scenario", scenario),
                "scores": session_scores
            }
        
        except Exception as e:
            logger.error(f"Error in run_task: {e}", exc_info=True)
            return {
                "session_id": "error",
                "num_turns": 0,
                "conversation_log": [],
                "error": str(e)
            }
    
    return run_task

if __name__ == "__main__":
    logger.info("Starting session-level conversation experiment")
    
    # Get dataset from Langfuse
    dataset_name = config.get('dataset', {}).get('name', 'simulated-conversations')
    logger.info(f"📥 Loading dataset: {dataset_name}")
    
    try:
        dataset = langfuse.get_dataset(dataset_name)
        logger.info(f"✓ Dataset loaded with {len(dataset.items)} items")
        
        # Run experiment using dataset.run_experiment
        result = dataset.run_experiment(
            name="session-evaluation-experiment",
            description="Multi-turn conversation evaluation with session-level scoring",
            task=create_task_function(langfuse, evaluator),
            max_concurrency=1
        )
        
        logger.info("\n" + result.format())
        
        # Final flush to ensure all data is sent
        logger.info("⏳ Final flush of all experiment data...")
        langfuse.flush()
        
        logger.info("✅ Experiment complete!")
        logger.info(f"📊 Check the Langfuse UI for detailed results")
        logger.info(f"   - Navigate to Sessions to view session-level scores")
        logger.info(f"   - Navigate to Scores to filter by session_id")
        logger.info(f"   - Navigate to Traces to see individual conversation turns")
        
    except Exception as e:
        logger.error(f"❌ Experiment failed: {e}", exc_info=True)
        sys.exit(1)

2026-03-18 16:40:06 - utils.utils2 - INFO - Found config at: c:\Users\ls3412\Langfuse_project\config\config.json
2026-03-18 16:40:06 - utils.utils2 - WARNING - Environment variable LANGFUSE_HOST not set, using placeholder
2026-03-18 16:40:06 - utils.utils2 - INFO - ✓ Configuration loaded from c:\Users\ls3412\Langfuse_project\config\config.json
2026-03-18 16:40:06 - __main__ - INFO - ✓ Configuration loaded successfully
2026-03-18 16:40:08 - utils.utils2 - INFO - ✓ Langfuse connection validated
2026-03-18 16:40:08 - __main__ - INFO - ✅ Langfuse client authenticated and ready
2026-03-18 16:40:08 - __main__ - INFO - ✓ Initialized 4 session evaluators
2026-03-18 16:40:08 - __main__ - INFO - ✓ Session evaluator initialized
2026-03-18 16:40:08 - __main__ - INFO - Starting session-level conversation experiment
2026-03-18 16:40:08 - __main__ - INFO - 📥 Loading dataset: 6-turn-travel-conversations
2026-03-18 16:40:08 - __main__ - INFO - ✓ Dataset loaded with 10 items
2026-03-18 16:40:09 - __main